<a href="https://colab.research.google.com/github/MihneaFeodot/Proiecte-Random/blob/PATR/BONUS_LAB_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [**Bonus**] Prioritatea în Threading

Într-un sistem ideal, am putea spune procesorului: *"Execută Thread-ul A cu prioritate maximă și Thread-ul B doar când ai timp liber"*.

Totuși, în Python, modulul `threading` **nu oferă un mecanism direct** pentru a seta prioritatea thread-urilor (cum ar fi `setPriority` în Java). Python lasă această decizie în totalitate pe seama "Planificatorului" (Scheduler) din sistemul de operare.



### Cum am putea simula prioritatea?

Propuneti un mecanism care ar putea implementa ideea de priorităti ale task-urilor

> **Hint**: se pot folosi diverse structuri de date




# Simularea Prioritatii Thread-urilor prin Work Stealing
Aceasta implementare compenseaza lipsa controlului direct asupra prioritatii thread-urilor in Python printr-o arhitectura asimetrica bazata pe cozi multiple si algoritmul de Work Stealing. Sistemul diferentiaza doua tipuri de executie: un Thread VIP (High Priority), care proceseaza prioritar sarcini din coada critica dar are capacitatea de a "fura" dinamic sarcini din coada inferioara pentru a maximiza utilizarea resurselor, si un Thread Normal, limitat strict la coada de prioritate scazuta. Aceasta logica garanteaza ca sarcinile urgente sunt finalizate primele, iar capacitatea de procesare a thread-ului prioritar este redistribuita automat pentru a ajuta la sarcinile lente, prevenind timpii morti (idle time).

In [ ]:
import threading
import queue
import time


High_queue = queue.Queue()
Low_queue = queue.Queue()

# Lista de cozi pentru a putea evidentia relatia de prioritate a workerilor
queues = [High_queue, Low_queue]

# WORKER VIP (rapid si are acces la ambele cozi)
def VIP_worker():
    while True:
        item = None
        source = ""

        # Prioritate 1: Verifica Coada High
        try:
            item = queues[0].get(timeout=0.05)
            source = "HIGH PRIORITY"
        except queue.Empty:
            # Prioritate 2: Daca High e gol, fura din Low
            try:
                item = queues[1].get(timeout=0.05)
                source = "LOW PRIORITY (STEALING)"
            except queue.Empty:
                continue # Ambele goale, mai incearca


        if item is None:
            # Trebuie anuntata coada corecta ca s-a terminat munca
            if source == "HIGH PRIORITY":
                queues[0].task_done()
            else:
                queues[1].task_done()
            break


        print(f"[VIP] Proceseaza: {item:<15} | Sursa: {source}")

        # VIP-ul are un timer mai mic decat cel al workerului normal pentru a asigura faptul ca si acesta se va ocupa de task-uri low priority
        time.sleep(0.1)

        if source == "HIGH PRIORITY":
            queues[0].task_done()
        else:
            queues[1].task_done()

    print("[VIP]    S-a oprit.")

# WORKER NORMAL (lent si are acces doar la coada low priority)
def Normal_worker():
    while True:
        item = queues[1].get() # Blocant pe Low_queue

        if item is None:
            queues[1].task_done()
            break

        print(f"[Normal] Proceseaza: {item:<15} | Sursa: LOW PRIORITY")

        time.sleep(0.3)
        queues[1].task_done()

    print("[Normal] S-a oprit.")


VIP_thread = threading.Thread(target=VIP_worker)
Normal_thread = threading.Thread(target=Normal_worker)

VIP_thread.start()
Normal_thread.start()

print("[Producator] adauga task-uri in coada")

# 20 task-uri Low Priority
for i in range(1, 21):
    queues[1].put(f"Task_Lent_{i}")

# 5 task-uri High Priority
for i in range(1, 6):
    queues[0].put(f"URGENT_{i}")


queues[0].join()
queues[1].join()

print("[Main] Toate task-urile utile au fost procesate. Trimitem semnalul de Stop.")


queues[0].put(None) # Pentru VIP


# doua semnale de stop pentru Normal, in caz ca pe unul il fura VIP-ul
queues[1].put(None)
queues[1].put(None)

VIP_thread.join()
Normal_thread.join()

[Producator] adauga task-uri in coada
[Normal] Proceseaza: Task_Lent_1     | Sursa: LOW PRIORITY
[VIP] Proceseaza: URGENT_1        | Sursa: HIGH PRIORITY
[VIP] Proceseaza: URGENT_2        | Sursa: HIGH PRIORITY
[VIP] Proceseaza: URGENT_3        | Sursa: HIGH PRIORITY
[Normal] Proceseaza: Task_Lent_2     | Sursa: LOW PRIORITY
[VIP] Proceseaza: URGENT_4        | Sursa: HIGH PRIORITY
[VIP] Proceseaza: URGENT_5        | Sursa: HIGH PRIORITY
[VIP] Proceseaza: Task_Lent_3     | Sursa: LOW PRIORITY (STEALING)
[Normal] Proceseaza: Task_Lent_4     | Sursa: LOW PRIORITY
[VIP] Proceseaza: Task_Lent_5     | Sursa: LOW PRIORITY (STEALING)
[VIP] Proceseaza: Task_Lent_6     | Sursa: LOW PRIORITY (STEALING)
[Normal] Proceseaza: Task_Lent_7     | Sursa: LOW PRIORITY
[VIP] Proceseaza: Task_Lent_8     | Sursa: LOW PRIORITY (STEALING)
[VIP] Proceseaza: Task_Lent_9     | Sursa: LOW PRIORITY (STEALING)
[Normal] Proceseaza: Task_Lent_10    | Sursa: LOW PRIORITY
[VIP] Proceseaza: Task_Lent_11    | Sursa: LOW 